# Somo la 10 - Wakala wa AI katika Uzalishaji

Katika somo hili utajifunza **mifumo ya uzalishaji** kwa mawakala wa AI ukitumia Microsoft Agent Framework (Python). Tunajumuisha:

- **Uwezekano wa kuangalia** — kuongeza muda na kufuatilia katika mwingiliano wa wakala
- **Tathmini** — kutumia wakala mtathmini kupima ubora wa majibu
- **Usimamizi wa gharama** — mikakati ya kuboresha tokeni na uchaguzi wa modeli

Hali ni ya **wakala wa kusafiria** anayesaidia watumiaji kupanga safari, huku ukifuatilia na kutathmini ukifika juu.


## Mipangilio


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Production Considerations

Moving AI agents from prototypes to production requires careful attention to three pillars:

1. **Observability** — You need visibility into what the agent is doing, how long it takes, and which tools it calls. Without tracing and logging, debugging production issues is nearly impossible.

2. **Evaluation** — Automated quality checks ensure the agent's responses remain accurate, complete, and helpful over time. An evaluator agent can score responses against defined criteria.

3. **Cost Management** — Token usage directly impacts cost. Strategies like prompt optimization, model selection, and caching help keep expenses under control without sacrificing quality.


## Kujenga Wakala Anayeonekana

Tunaelezea zana za usafiri na kufunika wito wa wakala na upigaji wa wakati ili tuweze kufuatilia ucheleweshaji. Katika utengenezaji ungeunganishwa na OpenTelemetry au mfumo wa uchunguzi kama huo.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mikakati ya Usimamizi wa Gharama

Kudhibiti gharama ni muhimu kwa mawakala wa AI wa utengenezaji. Hapa kuna mikakati muhimu:

| Mkakati | Maelezo |
|---|---|
| **Uboreshaji wa maagizo** | Weka maelekezo ya mfumo yafupi. Ondoa muktadha usiohitajika kupunguza tokeni za ingizo. |
    "| **Uchaguzi wa modeli** | Tumia modeli ndogo, nafuu (kwa mfano GPT-5-mini) kwa kazi rahisi kama uainishaji au uchanganuzi, na uhifadhi modeli kubwa kwa hoja ngumu. |\n",
| **Kuweka kwa hifadhidata** | Weka matokeo ya zana na maswali yanayojirudia ili kuepuka simu za API zisizohitajika. |
| **Mabusurute ya tokeni** | Weka mipaka ya `max_tokens` ili kuzuia majibu marefu yasiyotarajiwa. |
| **Kukusanya pamoja** | Kusanya maswali mengi ya mtumiaji katika simu moja ya API pale inavyowezekana. |

Katika utekelezaji, mbinu ya ngazi nyingi hufanya kazi vizuri: elekeza maombi rahisi kwa modeli ya haraka na nafuu na ongeza tu maswali ngumu kwa ile yenye uwezo zaidi.


## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kuongeza ufuatiliaji** kwa mwingiliano wa maajenti kwa kutumia muda na kurekodi, kuweka msingi wa kufuatilia na kuangalia.
2. **Kutathmini majibu ya maajenti** kiotomatiki kwa kutumia wakurugenzi wa tathmini ambao wanapima ukamilifu, usahihi, na msaada.
3. **Kudhibiti gharama** kupitia kuboresha maagizo, uchaguzi wa mfano, kuhifadhi taarifa, na bajeti za tokeni.

Mifumo hii ya uzalishaji husaidia kuhakikisha maajenti yako wa AI ni ya kuaminika, yanayopimika, na gharama nafuu kwa wingi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
